# Raikou Optical · Phase 0 · Notebook 3 — T4 parity, calibration, benchmark, demo, export

**Runtime: GPU → T4.** This is the same GPU as the g4dn production box (Turing sm_75, no bf16), so speed, memory and fp16 behaviour measured here carry over.

1. Calibrate a single temperature on the 15 **validation** tiles (whole-tile inference, T=1 and T=3).
2. Benchmark the 15 **test** tiles at T=1 and T=3, in fp32 and fp16, sieved and unsieved, against the unfiltered WorldCover-derived labels. These are **agreement** figures, not ground-truth accuracy.
3. Demo on the Visakhapatnam coast: a three-season stack from Planetary Computer (free), and optionally a single date from Copernicus (about 16.5 PU), each written as the full GIS bundle.
4. Export the release folder and tar for the g4dn box.

Time: about 1–1.5 h.

## 1 · Mount Drive, check the GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json
ROOT = '/content/drive/MyDrive/Raikou_Optical_Phase0'   # change only if you put the folder elsewhere
CODE = f'{ROOT}/optical_phase0'
assert os.path.exists(f'{CODE}/raikou_phase0/__init__.py'), f'Upload the optical_phase0 folder into {ROOT} first (see README).'
os.environ['RAIKOU_P0_ROOT'] = ROOT
if CODE not in sys.path:
    sys.path.insert(0, CODE)

import torch, numpy as np, pandas as pd
from IPython.display import Image, display
!nvidia-smi --query-gpu=name,memory.total --format=csv
if 'T4' not in torch.cuda.get_device_name(0):
    print('WARNING: not a T4. Results still valid, but the speed/memory numbers will not match the g4dn box.')

## 2 · Install

In [ ]:
!pip -q install olmoearth-pretrain-minimal==0.1.2 "odc-stac>=0.3.10" "pystac-client>=0.8" "planetary-computer>=1.0" "rasterio>=1.3.10" pyproj

## 3 · Load the manifest, the verified base config and the best checkpoint

In [ ]:
from tqdm.auto import tqdm
from raikou_phase0 import config as C, tiles, data as D, model as Mdl, evaluate_tiles as EV, calibrate as CAL, gis_export as G, selftest
paths = C.Paths().ensure()
tile_list = tiles.load_manifest(paths.manifest_csv)
by_id = {t.tile_id: t for t in tile_list}
val_ids = [t.tile_id for t in tile_list if t.split == 'val']
test_ids = [t.tile_id for t in tile_list if t.split == 'test']
base = Mdl.fetch_base_files('huggingface')
best_path = f'{paths.checkpoints}/best_model.pt'
payload = torch.load(best_path, map_location='cpu', weights_only=True)
model = Mdl.build_model(Mdl.load_base_encoder(base['base_dir'], load_weights=False), hparams=payload['hparams'])
Mdl.load_weights(model, best_path)
model = model.cuda().eval()
print('best epoch', payload['epoch'], '| patch size', payload['hparams']['patch_size'])
selftest.gpu_selftest(base['base_dir'], payload['hparams']['patch_size'])

## 4 · Copy the validation and test tiles to local disk

In [ ]:
D.stage_tiles(val_ids + test_ids, paths, progress=tqdm)

## 5 · Temperature scaling on the validation tiles (fp32)
Fitted against the **unfiltered** labels, so a calibrated score of 80 means about 80% of such pixels agreed with the WorldCover-derived class on held-out tiles.

In [ ]:
calib = EV.calibrate(model, val_ids, paths, amp='fp32', progress=tqdm)
CAL.reliability_png(f'{paths.reports}/reliability.png', {'ece': calib['ece_before'], 'bins': calib['bins_before']}, {'ece': calib['ece_after'], 'bins': calib['bins_after']}, calib['temperature'])
G.save_json(f'{paths.reports}/temperature_scale.json', calib)
print('temperature', calib['temperature'], '| ECE', calib['ece_before'], '->', calib['ece_after'])
display(Image(f'{paths.reports}/reliability.png'))

## 6 · Benchmark the 15 test tiles (fp32 and fp16)
fp16 is what the T4 box will run if the two agree. A mean-IoU difference above about 0.005 means production should stay in fp32.

In [ ]:
bench32 = EV.benchmark(model, test_ids, paths, amp='fp32', progress=tqdm)
bench16 = EV.benchmark(model, test_ids, paths, amp='fp16', progress=tqdm)
G.save_json(f'{paths.reports}/india_benchmark_metrics.json', {'fp32': bench32, 'fp16': bench16})
for m in ('T1', 'T3'):
    a, b = bench32['summary'][m], bench16['summary'][m]
    print(f"{m}: agreement {100 * a['overall_accuracy']:.1f}%  mIoU {a['mean_iou']:.3f}  macro F1 {a['macro_f1']:.3f}  | fp16 mIoU {b['mean_iou']:.3f} (delta {b['mean_iou'] - a['mean_iou']:+.4f})")
print('peak VRAM: fp32', bench32['peak_vram_gb'], 'GB, fp16', bench16['peak_vram_gb'], 'GB')
print('seconds per tile (fp16):', pd.DataFrame(bench16['timing']).groupby('mode').seconds.mean().round(1).to_dict())
display(pd.DataFrame({cls: v for cls, v in bench32['summary']['T3']['per_class'].items()}).T)

In [ ]:
card_stub = {'name': 'Raikou OlmoEarth India LULC (5 classes)', 'version': '0.1.0', 'calibration': {k: v for k, v in calib.items() if not k.startswith('bins')}}
DEMO_TILE = 'coast_visakhapatnam_1'
t = by_id[DEMO_TILE]
print(DEMO_TILE, t.crs, t.bounds)

## 7 · Demo A: three-season stack from Planetary Computer (free)
Oct 2025 → May 2026 for the Visakhapatnam coast tile, written as the full GIS bundle to `release_export/demo_visakhapatnam_T3/`.

In [ ]:
from raikou_phase0 import s2_pc, bundle
DEMO_HYDRO_YEAR = 2025
cat = s2_pc.open_catalog()
T = len(C.SEASONS)
dn = np.zeros((T, 12, t.height, t.width), np.uint16); valid = np.zeros((T, t.height, t.width), bool); dates = []
for i, w in enumerate(C.season_windows(DEMO_HYDRO_YEAR)):
    acq = s2_pc.pick_acquisition(cat, t, w['start'], w['end'])
    if acq is None:
        dates.append(w['nominal']); continue
    got = s2_pc.load_acquisition(acq, t)
    dn[i], valid[i] = got['dn'], got['valid']; dates.append(acq['date'])
    print(w['name'], acq['date'], 'clear', round(got['valid_fraction'], 3), 'baseline', got['baseline'], 'offset removed', got['offset_applied'])
use = [i for i in range(T) if valid[i].mean() >= C.MIN_TIMESTEP_VALID] or [int(valid.reshape(T, -1).mean(1).argmax())]
summ = bundle.run_aoi(model, t, dn, valid, dates, f'{paths.release}/demo_visakhapatnam_T3', 'visakhapatnam', card_stub, bench32, temperature=calib['temperature'], use_timesteps=use, amp='fp16')
print(summ)
for n in ('true_color', 'lulc', 'worldcover2021', 'model_view_pca', 'similarity'):
    display(Image(f"{summ['out_dir']}/visakhapatnam_{n}.png", width=420))

## 8 · Demo B (optional): one date from Copernicus, exactly as production will fetch it
About **16.5 PU**. Needs the Colab secrets `CDSE_CLIENT_ID` and `CDSE_CLIENT_SECRET`. Planetary Computer is only used to choose the clearest date (free); the pixels come from Copernicus. Set `RUN_LIVE = True` to run it.

In [ ]:
RUN_LIVE = False
LIVE_WINDOW = ('2026-04-01', '2026-05-20')   # pick a recent clear window; Oct-Nov is often cloudy until the monsoon ends
if RUN_LIVE:
    from google.colab import userdata
    from raikou_phase0 import cdse, s2_pc, bundle
    print('Estimated cost:', round(cdse.estimate_pu(t.width, t.height), 1), 'PU')
    acq = s2_pc.pick_acquisition(s2_pc.open_catalog(), t, *LIVE_WINDOW)
    token = cdse.get_token(userdata.get('CDSE_CLIENT_ID'), userdata.get('CDSE_CLIENT_SECRET'))
    got = cdse.fetch(t, token, *cdse.window_around(acq['datetime']))
    print('date', acq['date'], '| PU spent', got['pu_spent'], '| clear', round(got['valid_fraction'], 3))
    summ_live = bundle.run_aoi(model, t, got['dn'][None], got['valid'][None], [acq['date']], f'{paths.release}/demo_visakhapatnam_T1_cdse', 'visakhapatnam', card_stub, bench32, temperature=calib['temperature'], amp='fp16')
    print(summ_live)
    display(Image(f"{summ_live['out_dir']}/visakhapatnam_lulc.png", width=420))

Download a demo folder from Drive and drag `visakhapatnam_lulc_10m.tif` into QGIS: it should open with the five class colours and names (the `.qml` next to it is loaded automatically). Check that there are no seams, that the shoreline lines up with a basemap, and that the polygons in the `.gpkg` have areas.

## 9 · Export the release for the g4dn box

In [ ]:
from raikou_phase0 import release, infer
manifest_summary = tiles.check_manifest(tile_list)['per_zone']
out = release.export(paths, best_path, base, bench32, bench16, calib, manifest_summary, f'{CODE}/raikou_phase0')
print('release folder:', out)
!ls -la "{out}"
!cat "{paths.release}/olmoearth_india_v1_2.tar.sha256"

# the backend contract, loaded from the release exactly as the box will load it
m2 = infer.load_model(f'{out}/olmoearth_india_lulc_fp16.pt', f'{out}/model_card.json')
lt = D.LocalTile(paths.local, test_ids[0])
probs, emb = infer.predict(m2, np.asarray(lt.arr('dn')), np.asarray(lt.arr('valid')).astype(bool), lt.dates)
print('predict():', probs.shape, probs.dtype, '| embeddings', emb.shape, emb.dtype)

## 10 · Move it to the g4dn box
The tar is about 1 GB (fp32 + fp16 weights). On the box, pull it from Drive and check the hash (README → *Moving the release to the g4dn box*). Do not unpack it into the SAR backend's venv: per the approved plan, the optical model runs in its own venv and worker (Phase 1–2).